# 02 — Features and Encoding

This notebook builds on the cleaned dataset from `01_eda_and_cleaning.ipynb` (loaded directly from
`clean.parquet`, not re-derived) and looks at:

1. How `sneakerml.features.parse_sneaker_name` turns a hyphenated StockX-style name into structured
   catalog fields.
2. A genuine, live 3-way comparison of categorical-encoding strategies on a 10k-row subsample.
3. Why `OneHotEncoder(handle_unknown="ignore")` matters, with a small deliberate repro.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from sneakerml.config import PROCESSED_DIR, RANDOM_SEED
from sneakerml.features import CATEGORICAL, NUMERIC, TARGET, add_features, parse_sneaker_name

pd.set_option("display.max_columns", 20)
clean_df = pd.read_parquet(PROCESSED_DIR / "clean.parquet")
clean_df.shape

(90944, 11)

## 1. `parse_sneaker_name` on real names

A few random real names from the clean dataset, plus a couple of hand-picked "interesting" cases
(an Off-White collab, and a bare Yeezy) to show the collab/base-brand detection logic.

In [2]:
sample_names = clean_df["sneaker_name"].sample(5, random_state=RANDOM_SEED).tolist()
interesting_names = [
    "Air-Jordan-1-Retro-High-Off-White-Chicago",
    "Adidas-Yeezy-Boost-350-V2-Zebra",
]
demo_names = sample_names + interesting_names

parsed_demo = pd.DataFrame(
    [{"sneaker_name": n, **parse_sneaker_name(n)} for n in demo_names]
)
parsed_demo

,sneaker_name,base_brand,silhouette,colorway,collab_partner,is_collab
0,Adidas-Yeezy-Boost-350-V2-Core-Black-Red-2017,Adidas,Yeezy Boost 350 V2,Core Black Red 2017,Yeezy,True
1,Adidas-Yeezy-Boost-350-V2-Cream-White,Adidas,Yeezy Boost 350 V2,Cream White,Yeezy,True
2,Adidas-Yeezy-Boost-350-V2-Zebra,Adidas,Yeezy Boost 350 V2,Zebra,Yeezy,True
3,Nike-Blazer-Mid-Off-White-All-Hallows-Eve,Nike,Blazer Mid,All Hallows Eve,Off-White,True
4,Air-Jordan-1-Retro-High-Off-White-University-Blue,Nike,Air Jordan 1,University Blue,Off-White,True
5,Air-Jordan-1-Retro-High-Off-White-Chicago,Nike,Air Jordan 1,Chicago,Off-White,True
6,Adidas-Yeezy-Boost-350-V2-Zebra,Adidas,Yeezy Boost 350 V2,Zebra,Yeezy,True


Note how `Off-White` is detected as the `collab_partner` (with `base_brand` forced to `Nike`)
purely from the name text, and `Yeezy` maps `base_brand` to `Adidas` even though the raw StockX
`brand` column already says "Adidas" for that row — the parser derives structure independently of
the (sometimes messy) upstream `brand` field.

## 2. Build model-ready features

`add_features` is the same function used by training and by `PricePredictor` at serving time —
calling it here (rather than reimplementing it) keeps this notebook honest about what the model
actually sees.

In [3]:
featured = add_features(clean_df)
featured = featured.dropna(subset=[TARGET])
print(featured.shape)
featured[CATEGORICAL + NUMERIC + [TARGET]].head(3)

(90944, 20)


,brand,base_brand,silhouette,collab_partner,colorway,buyer_region,size_bucket,source,shoe_size,retail_price,days_since_release,is_half_size,size_imputed,sale_price
0,Yeezy,Adidas,Yeezy Boost 350 Low,Yeezy,Moonrock,Michigan,core,stockx,10.5,200.0,663,True,False,1230.0
1,Yeezy,Adidas,Yeezy Boost 350 Low,Yeezy,Moonrock,Nevada,core,stockx,9.5,200.0,682,True,False,1257.0
2,Yeezy,Adidas,Yeezy Boost 350 Low,Yeezy,Moonrock,California,small,stockx,7.0,200.0,690,False,False,1000.0


## 3. A 10k-row subsample for fast, live model fits

The full ~91k-row dataset is unnecessary for an *encoding-strategy* comparison — a 10k-row random
subsample is plenty to see the relative behaviour of one-hot vs. ordinal vs. native-categorical
handling, and keeps this notebook's runtime small.

In [4]:
SUBSAMPLE_N = 10_000
sub = featured.sample(n=SUBSAMPLE_N, random_state=RANDOM_SEED)

X = sub[CATEGORICAL + NUMERIC]
y = np.log1p(sub[TARGET].to_numpy(dtype=float))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
X_train.shape, X_test.shape

((8000, 13), (2000, 13))

## 4. Three encoding strategies, one regressor family

We hold the regressor family fixed (`HistGradientBoostingRegressor`, a fast tree ensemble) and vary
only how the categorical columns are represented, so any MAE differences are attributable to the
encoding rather than to a different model class:

- **(a) One-hot** — `OneHotEncoder(handle_unknown="ignore")` fit on the training split, transformed
  into a dense matrix, fed to `HistGradientBoostingRegressor` as plain numeric columns.
- **(b) Ordinal** — `OrdinalEncoder` fit the same way; each category becomes a single integer code.
  This is a cheaper representation but silently implies a false ordering between categories (is
  "Yeezy Boost 350" really "less than" "Air Jordan 1" in some meaningful sense? No — but an ordinal
  code lets the tree pretend there is one).
- **(c) Native categorical** — `HistGradientBoostingRegressor`'s own `categorical_features` support
  (its default, `"from_dtype"`, is used here explicitly): the categorical columns are cast to
  pandas `category` dtype and handed to the model with *no* encoding step at all. The tree learns
  arbitrary category-subset splits internally, which is both cheaper to fit and free of the
  ordinal encoder's false-ordering assumption.

All three use `max_iter=50` — small on purpose, since this is a comparison of encodings, not a
search for the best possible model (the shipped model in notebook 03 uses far more iterations with
early stopping).

In [5]:
MAX_ITER = 50
results = []

# (a) one-hot
t0 = time.perf_counter()
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
Xtr_ohe = ohe.fit_transform(X_train[CATEGORICAL])
Xte_ohe = ohe.transform(X_test[CATEGORICAL])
Xtr_ohe = np.hstack([Xtr_ohe, X_train[NUMERIC].to_numpy(dtype=float)])
Xte_ohe = np.hstack([Xte_ohe, X_test[NUMERIC].to_numpy(dtype=float)])
model_ohe = HistGradientBoostingRegressor(max_iter=MAX_ITER, random_state=RANDOM_SEED)
model_ohe.fit(Xtr_ohe, y_train)
pred_ohe = model_ohe.predict(Xte_ohe)
mae_ohe = mean_absolute_error(np.expm1(y_test), np.expm1(pred_ohe))
fit_time_ohe = time.perf_counter() - t0
results.append({"encoding": "one-hot", "n_features": Xtr_ohe.shape[1], "mae_usd": mae_ohe, "fit_seconds": fit_time_ohe})

# (b) ordinal
t0 = time.perf_counter()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
Xtr_ord = oe.fit_transform(X_train[CATEGORICAL])
Xte_ord = oe.transform(X_test[CATEGORICAL])
Xtr_ord = np.hstack([Xtr_ord, X_train[NUMERIC].to_numpy(dtype=float)])
Xte_ord = np.hstack([Xte_ord, X_test[NUMERIC].to_numpy(dtype=float)])
model_ord = HistGradientBoostingRegressor(max_iter=MAX_ITER, random_state=RANDOM_SEED)
model_ord.fit(Xtr_ord, y_train)
pred_ord = model_ord.predict(Xte_ord)
mae_ord = mean_absolute_error(np.expm1(y_test), np.expm1(pred_ord))
fit_time_ord = time.perf_counter() - t0
results.append({"encoding": "ordinal", "n_features": Xtr_ord.shape[1], "mae_usd": mae_ord, "fit_seconds": fit_time_ord})

# (c) native categorical (HGB's own from_dtype handling, no encoder at all)
t0 = time.perf_counter()
X_train_native = X_train.copy()
X_test_native = X_test.copy()
for c in CATEGORICAL:
    X_train_native[c] = X_train_native[c].astype("category")
    X_test_native[c] = pd.Categorical(X_test_native[c], categories=X_train_native[c].cat.categories)
model_native = HistGradientBoostingRegressor(
    max_iter=MAX_ITER, random_state=RANDOM_SEED, categorical_features="from_dtype"
)
model_native.fit(X_train_native, y_train)
pred_native = model_native.predict(X_test_native)
mae_native = mean_absolute_error(np.expm1(y_test), np.expm1(pred_native))
fit_time_native = time.perf_counter() - t0
results.append({"encoding": "native categorical", "n_features": X_train_native.shape[1], "mae_usd": mae_native, "fit_seconds": fit_time_native})

results_df = pd.DataFrame(results)
results_df

,encoding,n_features,mae_usd,fit_seconds
0,one-hot,122,37.531736,0.275477
1,ordinal,13,37.278800,0.164227
2,native categorical,13,35.129996,0.174476


One-hot inflates the feature count by roughly the number of distinct category levels (over
50 `buyer_region` values alone), while ordinal and native-categorical stay at the original column
count. Native categorical handling tends to fit fastest and often matches or beats one-hot's MAE on
tree models, since it avoids both the dimensionality blow-up of one-hot *and* the false ordering
implied by plain ordinal codes — trees can split on arbitrary category subsets directly.

The shipped model (`sneakerml.train.build_pipeline`) actually uses one-hot encoding rather than
native categorical support. That's a defensible, if not the highest-raw-accuracy, choice: one-hot
plus `min_frequency` grouping keeps the encoding easy to introspect column-by-column, which is
exactly what `sneakerml.explain.Explainer` relies on to attribute SHAP values back to original
feature names (`silhouette`, `colorway`, ...) rather than opaque per-tree-split categorical rules.

## 5. `handle_unknown="ignore"` — why it matters

`buyer_region` has some very rare values (Wyoming, Montana, Alaska, ...). If a rare region happens
to land entirely in the test split and never appears in training, a `OneHotEncoder` fit *without*
`handle_unknown="ignore"` will raise at transform time. With it, the unseen category just encodes
as an all-zero row instead of crashing.

We reproduce this directly: fit an encoder on a training split with one rare region deliberately
held out, then transform a test split that includes it.

In [6]:
region_counts = X_train["buyer_region"].value_counts()
rare_region = region_counts.index[-1]
print(f"rare region held out of a synthetic 'training' fit: {rare_region!r}")

train_without_rare = X_train[X_train["buyer_region"] != rare_region]
test_with_rare = X_test[X_test["buyer_region"] == rare_region]
if len(test_with_rare) == 0:
    # guarantee at least one row carries the held-out region so the repro is real
    test_with_rare = X_test.iloc[[0]].copy()
    test_with_rare["buyer_region"] = rare_region

print(f"rows in the held-out training fit: {len(train_without_rare)}")
print(f"test rows carrying the never-seen region: {len(test_with_rare)}")

rare region held out of a synthetic 'training' fit: 'Montana'
rows in the held-out training fit: 7999
test rows carrying the never-seen region: 1


In [7]:
strict_encoder = OneHotEncoder(handle_unknown="error")
strict_encoder.fit(train_without_rare[CATEGORICAL])
try:
    strict_encoder.transform(test_with_rare[CATEGORICAL])
    print("no error raised (unexpected)")
except ValueError as e:
    print("OneHotEncoder(handle_unknown='error') raised, as expected:")
    print(f"  {e}")

OneHotEncoder(handle_unknown='error') raised, as expected:
  Found unknown categories ['Montana'] in column 5 during transform


In [8]:
safe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
safe_encoder.fit(train_without_rare[CATEGORICAL])
transformed = safe_encoder.transform(test_with_rare[CATEGORICAL])
print("OneHotEncoder(handle_unknown='ignore') succeeds; unseen category encodes as all-zeros.")
print("row sum for the buyer_region one-hot block should be 0 for the held-out region's rows.")
print(transformed.sum())  # sanity: transform succeeded and returned a real matrix

OneHotEncoder(handle_unknown='ignore') succeeds; unseen category encodes as all-zeros.
row sum for the buyer_region one-hot block should be 0 for the held-out region's rows.
7.0


This is exactly why `sneakerml.train.build_pipeline` uses `handle_unknown="ignore"`: a
`buyer_region` or rare `colorway` that shows up in production but never appeared during training
must not crash the price predictor — it should just fall back to "no signal from this category"
and let the other features carry the prediction.

## Summary

- `parse_sneaker_name` extracts `base_brand`, `silhouette`, `colorway`, `collab_partner` directly
  from the hyphenated name text, independent of the (sometimes inconsistent) upstream `brand`
  field.
- On a 10k-row subsample, one-hot, ordinal, and native-categorical encodings were compared under an
  identical `HistGradientBoostingRegressor(max_iter=50)` — see the results table above for the real
  MAE/fit-time numbers from this run.
- `OneHotEncoder(handle_unknown="ignore")` is not a defensive nicety here — a real column
  (`buyer_region`) has several categories rare enough that a training/serving split without it
  would eventually crash the model at prediction time.